# Coding Encoder-Decoder Attention and Multi-Head Attention

## Encoder-Decoder Attention

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [5]:
class Attention(nn.Module):
    def __init__(self, d_model, row_dim, col_dim):
        super().__init__()
        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        
        self.row_dim = row_dim
        self.col_dim = col_dim
    
    def forward(self, encoding_for_q, encoding_for_k, encoding_for_v, mask=None):
        q = self.W_q(encoding_for_q)
        k = self.W_k(encoding_for_k)
        v = self.W_v(encoding_for_v)
        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))
        scaled_sims = sims/torch.tensor(k.size(self.col_dim)**0.5)
        
        if mask is not None:
            scaled_sims = scaled_sims.masked_fill(mask, value=-1e9)
        
        attention_proportions = F.softmax(scaled_sims)
        attention_scores = torch.matmul(attention_proportions, v)
        return attention_scores

In [6]:
# Creating matrices for token encodings
# Encodings for Q comes from decoder and Encodings for K and V comes from encoder, so I am using the attention values came from the previous two notebooks
encoding_for_q = torch.tensor([[ 0.6038,  0.7434],
                               [-0.0062,  0.6072],
                               [ 3.4989,  2.2427]])

encoding_for_k = torch.tensor([[1.0100, 1.0641],
                               [0.2040, 0.7057],
                               [3.4989, 2.2427]])

encoding_for_v = torch.tensor([[1.0100, 1.0641],
                               [0.2040, 0.7057],
                               [3.4989, 2.2427]]) # Same as k

torch.manual_seed(42)

attention = Attention(d_model=2, row_dim=0, col_dim=1)

attention(encoding_for_q, encoding_for_k, encoding_for_v, mask=None)

C:\Users\samoj\AppData\Local\Temp\ipykernel_32028\3537292704.py:21: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  attention_proportions = F.softmax(scaled_sims)


tensor([[0.2218, 1.0295],
        [0.2388, 1.0597],
        [0.0963, 0.8067]], grad_fn=<MmBackward0>)

## Multi-Head Attention

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=2, row_dim=0, col_dim=1, num_heads=1):
        super().__init__()
        self.heads = nn.ModuleList(
            [Attention(d_model, row_dim, col_dim) for _ in range(num_heads)]
        )
        self.col_dim = col_dim
    
    def forward(self, encoding_for_q, encoding_for_k, encoding_for_v):
        return torch.cat([head(encoding_for_q, encoding_for_k, encoding_for_v) for head in self.heads], dim=self.col_dim)

In [ ]:
torch.manual_seed(42)

multihead_attention = MultiHeadAttention(d_model=2, row_dim=0, col_dim=1, num_heads=1)

multihead_attention(encoding_for_q, encoding_for_k, encoding_for_v) # Since num_heads = 1, we should get the same values as above

C:\Users\samoj\AppData\Local\Temp\ipykernel_32028\3537292704.py:21: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  attention_proportions = F.softmax(scaled_sims)


tensor([[0.2218, 1.0295],
        [0.2388, 1.0597],
        [0.0963, 0.8067]], grad_fn=<CatBackward0>)

In [9]:
multihead_attention = MultiHeadAttention(d_model=2, row_dim=0, col_dim=1, num_heads=3)

multihead_attention(encoding_for_q, encoding_for_k, encoding_for_v)

C:\Users\samoj\AppData\Local\Temp\ipykernel_32028\3537292704.py:21: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  attention_proportions = F.softmax(scaled_sims)


tensor([[-0.7270,  0.1569,  1.0106,  0.2239,  0.2209,  0.7966],
        [-0.6474,  0.1716,  0.9809,  0.2252,  0.2120,  0.7810],
        [-1.1023,  0.0879,  1.1082,  0.2199,  0.2401,  0.8306]],
       grad_fn=<CatBackward0>)

Note:
- When calculating the encoder-decoder attention, we know that encoder will have its own Q, K, and V; and decoder will also have its own Q, K, and V. While Encoder uses Self-Attention (or Multi-head attention) and Decoder uses Masked-Self-Attention (Masked-Multi-head attention), the encoder-decoder attention will have a new set of Q, K, and V that are not a part of the ones that are present in Encoder and Decoder, this is a new set, where the Query Weights come from the Decoder and a new set of weights for K and V will come from Encoder side while calculating this new Encoder-Decoder Attention values.
- To put the above point simply: The sets of Weights that we use to calculate the Queries, Keys, and Values for Encoder-Decoder Attention are different from the sets of Weights we use for Self-Attention. However, just like for Self-Attention, the sets of Weights are copied and reused for each word.
- Remember, when calculating the positional encoding, for each dimension (x, y, z, or the ones that come after), the curve used will be the same for both encoder and decoder. Let's say, we are using Sin for x, Cos for y, Sin2A for z and so on... We will use the same Sin for x of decoder and so on... Remember, there is significance of the value of x to find the positional encoding that means for the first word the positional encoding value that will be added to the token embedding will be 0 irrespective of the value present in the x axis of the token embedding.

Here I am talking about the Residual connections.
- We create Residual connections on top of the self-attention values for both encoder and decoder. How? By just adding the positional encoded values (token embeddings + positional encodings) with the self-attention (or masked self-attention for decoder) and get residual connections. When creating the Encoder-Decoder Attention, we actually use the values from Residual connections rather than direct Self-Attention values.
- Do we create Residual connections for Encoder-Decoder Attention as well? If yes, which values will be added to Encoder-Decoder Attention values?
    - Yes! For Encoder-Decoder Attention values, we add the self-attention values of the Decoder to Encoder-Decoder Attention to get Residual connections. Why?
    - The residual connection for Encoder-Decoder Attention allows the decoder to focus on the relationship between output tokens and input tokens (from the encoder) while preserving the original information from the decoder's previous layers (self-attention). It helps the model retain both the input context (from the encoder) and the context built in the decoder, without overwriting earlier self-attention or positional encodings.
- Just like we can create multi-head attention in the case of encoder and decoder self-attention or masked self-attention steps, we can do it for encoder-decoder attention values. How? We do the same stacking of Q, K, and V like we do in the case of Encoders and Decoders.
- What do we use this residual connection of encoder-decoder attention?
    - If we consider a translation task, we will use this as the input for the fully connected neural network. The input size equals the size of the residual connection and the output size is the number of tokens in the input. After training and passing through softmax, each residual input from the decoder will have one token matching from the input (input text).
    - In the intial stages during the training process there will be misclassifications and with backpropagation like every other neural network, they will get adjusted and eventually match the appropriate tokens and the training ends.